Размеры матриц: (100, 1000, 10000) х (4, 20, 100), полноранговые гауссовые разложение: 

RandomizedSVD x ( 'normal',
    'ortho',
    'sparse_iid_entries',
    'identity_copies',)
    
перебрать ранги (метод decompose, параметр rank) из (0.05, 0.25, 0.5)

In [ ]:
import sys
import os
from typing import List, Dict, Tuple, Callable, Any
import torch
import time
import numpy as np
import matplotlib.pyplot as plt
import logging
from datetime import datetime
import pandas as pd

In [ ]:
from tdecomp.matrix.decomposer import RandomizedSVD

In [ ]:
def run_benchmarks_cuda():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    inits = ['normal', 'ortho', 'sparse_iid_entries', 'identity_copies']
    ranks = [0.05, 0.25, 0.5, 0.8, 0.9, 1]
    results = []

    for M in (4, 20, 100):
        for N in (100, 1000, 10000):
            W = torch.randn(M, N, device=device)
            
            for init_type in inits:
                svd_model = RandomizedSVD(random_init=init_type)
                
                for r in ranks:
                    times = []
                    errors = []
                    
                    for _ in range(5):
                        start_event = torch.cuda.Event(enable_timing=True)
                        end_event = torch.cuda.Event(enable_timing=True)
                        
                        start_event.record()
                        factors = svd_model.decompose(W, rank=r)
                        end_event.record()
                        torch.cuda.synchronize()
                        elapsed_time_ms = start_event.elapsed_time(end_event)
                        times.append(elapsed_time_ms)
                        
                        error = svd_model.get_approximation_error(W, *factors)
                        errors.append(error.item())
                    
                    results.append({
                        "M": M,
                        "N": N,
                        "Init": init_type,
                        "Rank": r,
                        "Avg_Time_ms": sum(times) / len(times),
                        "Avg_Error": sum(errors) / len(errors)
                    })
                    
            torch.cuda.empty_cache()

    return pd.DataFrame(results)


df_results = run_benchmarks_cuda()
df_results


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df = df_results

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df_err = df[df['M'] == 100]
sns.lineplot(
    ax=axes[0], data=df_err, 
    x='Rank', y='Avg_Error', hue='N', 
    marker='o', palette='viridis'
)
axes[0].set_title('Approximation Error vs Rank (M=100)')
axes[0].set_ylabel('Relative Frobenius Norm Error')
axes[0].set_xlabel('Rank')

sns.lineplot(
    ax=axes[1], data=df_err, 
    x='Rank', y='Avg_Time_ms', hue='N', 
    marker='o', palette='viridis'
)

axes[1].set_title('Execution Time vs Rank (M=100)')
axes[1].set_ylabel('Time (ms)')
axes[1].set_xlabel('Rank')

plt.tight_layout()
plt.show()